In [3]:
import pandas as pd
import numpy as np

# Chia dữ liệu
from sklearn.model_selection import train_test_split

# Xử lý dữ liệu
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Mô hình
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

# Đánh giá mô hình
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Lưu model
import joblib

In [4]:
# Đọc dữ liệu đã làm sạch
df = pd.read_csv("../data/processed/houses_clean.csv")

# Kiểm tra kích thước
print("Kích thước dữ liệu:", df.shape)

# Xem dữ liệu
display(df.head())

Kích thước dữ liệu: (9999, 19)


,name,description,property_type_name,province_name,district_name,ward_name,street_name,project_name,price,area,floor_count,frontage_width,house_depth,road_width,bedroom_count,bathroom_count,house_direction,balcony_direction,published_at
0,"Bán căn hộ dịch vụ 40 phòng, lợi nhuận ổn định...",Bán căn hộ dịch vụ mặt tiền 164 - 166 Nguyễn C...,Nhà,Hồ Chí Minh,Tân Phú,Tân Sơn Nhì,Nguyễn Cửu Đàm,Không rõ,3.500000e+10,130.0,3.0,8.0,15.0,6.0,3.0,2.0,Đông Bắc,Đông Bắc,2025-06-26T11:24:18
1,"Nguyễn Ngọc Lộc Q10, 5tầng BTCT hẻm 5m thông, ...",NGUYỄN NGỌC LỘC - HẺM XE HƠI 5M THÔNG - NHÀ 5 ...,Nhà,Hồ Chí Minh,10,14,Nguyễn Ngọc Lộc,Không rõ,6.500000e+09,24.0,5.0,3.7,15.0,6.0,3.0,4.0,Không rõ,Không rõ,2025-11-17T15:36:52.649000
2,"Nhà hẻm xe hơi ngay Aeon Mall Tân Phú, diện tí...",Nhà siêu hot ngay Aeon Mall Tân Phú đây! Diện ...,Nhà,Hồ Chí Minh,11,Không rõ,Không rõ,Không rõ,6.000000e+09,124.0,3.0,4.0,31.0,6.0,3.0,2.0,Không rõ,Không rõ,2025-09-16T09:56:07.702000
3,"Bán chung cư 4PN, 4WC tại Tôn Thất Thuyết, 64m...","Căn hộ chung cư nằm tại Đường Tôn Thất Thuyết,...",Căn hộ chung cư,Hồ Chí Minh,4,1,Tôn Thất Thuyết,Không rõ,7.500000e+09,64.0,3.0,4.9,15.0,6.0,4.0,4.0,Không rõ,Không rõ,2026-03-20T21:24:50.219000
4,"Bán gấp CC Tây Thạnh DT 72m2, 2 tỷ 7 TL 2PN 2W...","Bán chung cư Tây Thạnh, P. Tây Thạnh, Q. Tân P...",Căn hộ chung cư,Hồ Chí Minh,Tân Phú,Tây Thạnh,C8,Không rõ,2.700000e+09,72.0,3.0,4.9,15.0,6.0,2.0,2.0,Không rõ,Không rõ,2025-10-17T09:40:43.092000


In [5]:
# Tạo tổng số phòng
df["total_rooms"] = (
    df["bedroom_count"] +
    df["bathroom_count"]
)

# Tạo diện tích trung bình mỗi phòng
df["area_per_room"] = (
    df["area"] /
    df["total_rooms"].replace(0, 1)
)

# Chuyển ngày đăng sang kiểu datetime
df["published_at"] = pd.to_datetime(
    df["published_at"],
    errors="coerce"
)

# Tạo năm đăng tin
df["published_year"] = df["published_at"].dt.year

# Tạo tháng đăng tin
df["published_month"] = df["published_at"].dt.month

# Tạo thứ trong tuần
df["published_dayofweek"] = (
    df["published_at"].dt.dayofweek
)

# Độ dài mô tả
df["description_length"] = (
    df["description"].fillna("").str.len()
)

# Độ dài tên tin
df["name_length"] = (
    df["name"].fillna("").str.len()
)

print("Đã tạo Feature.")

Đã tạo Feature.


In [6]:
# xác định biến mục tiêu 
# Biến mục tiêu
y = df["price"]

# Kiểm tra
print("Biến mục tiêu: price")
print("Số lượng:", len(y))

Biến mục tiêu: price
Số lượng: 9999


In [7]:
# tách x và y 
# Các cột văn bản
text_cols = [
    "name",
    "description"
]

# Các cột dạng phân loại
categorical_cols = [
    "property_type_name",
    "province_name",
    "district_name",
    "ward_name",
    "street_name",
    "project_name",
    "house_direction",
    "balcony_direction"
]

# Các cột dạng số
numeric_cols = [
    "area",
    "floor_count",
    "frontage_width",
    "house_depth",
    "road_width",
    "bedroom_count",
    "bathroom_count",
    "total_rooms",
    "area_per_room",
    "published_year",
    "published_month",
    "published_dayofweek",
    "description_length",
    "name_length"
]

print("Số cột văn bản:", len(text_cols))
print("Số cột phân loại:", len(categorical_cols))
print("Số cột số:", len(numeric_cols))

Số cột văn bản: 2
Số cột phân loại: 8
Số cột số: 14


In [8]:
# chuẩn bị dữ liệu
# Tạo bản sao
data = df.copy()

# Điền dữ liệu thiếu cho cột văn bản
for col in text_cols:
    data[col] = data[col].fillna("")

# Điền dữ liệu thiếu cho cột phân loại
for col in categorical_cols:
    data[col] = data[col].fillna("Không rõ")

# Điền dữ liệu thiếu cho cột số
for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

print("Đã xử lý dữ liệu đầu vào.")

Đã xử lý dữ liệu đầu vào.


In [9]:
# Tạo X gồm tất cả đặc trưng
X = data[
    text_cols +
    categorical_cols +
    numeric_cols
].copy()

# Kiểm tra
print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

Kích thước X: (9999, 24)
Kích thước y: (9999,)


In [10]:
# Chia 80% train và 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Kiểm tra
print("Số dòng Train:", len(X_train))
print("Số dòng Test:", len(X_test))

Số dòng Train: 7999
Số dòng Test: 2000


In [11]:
# tạo bộ xử lý dữ liệu
# Xử lý cột name bằng TF-IDF
name_vectorizer = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2)
)

# Xử lý description bằng TF-IDF
description_vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2)
)

# One-Hot Encoding cho dữ liệu phân loại
onehot = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Bộ xử lý toàn bộ dữ liệu
preprocessor = ColumnTransformer(
    transformers=[
        (
            "name_tfidf",
            name_vectorizer,
            "name"
        ),
        (
            "description_tfidf",
            description_vectorizer,
            "description"
        ),
        (
            "categorical",
            onehot,
            categorical_cols
        ),
        (
            "numeric",
            "passthrough",
            numeric_cols
        )
    ]
)

print("Đã tạo bộ tiền xử lý.")

Đã tạo bộ tiền xử lý.


In [12]:
# ==============================
# 1. LINEAR REGRESSION
# ==============================

model_lr = LinearRegression()

pipeline_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model_lr)
    ]
)

print("Đang huấn luyện Linear Regression...")

pipeline_lr.fit(
    X_train,
    y_train
)

print("Huấn luyện Linear Regression hoàn tất.")

Đang huấn luyện Linear Regression...
Huấn luyện Linear Regression hoàn tất.


In [13]:
# ==============================
# 2. RANDOM FOREST
# ==============================

model_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

pipeline_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model_rf)
    ]
)

print("Đang huấn luyện Random Forest...")

pipeline_rf.fit(
    X_train,
    y_train
)

print("Huấn luyện Random Forest hoàn tất.")

Đang huấn luyện Random Forest...
Huấn luyện Random Forest hoàn tất.


In [14]:
# ==============================
# 3. XGBOOST
# ==============================

model_xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipeline_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model_xgb)
    ]
)

print("Đang huấn luyện XGBoost...")

pipeline_xgb.fit(
    X_train,
    y_train
)

print("Huấn luyện XGBoost hoàn tất.")

Đang huấn luyện XGBoost...
Huấn luyện XGBoost hoàn tất.


In [15]:
# ==============================
# DỰ ĐOÁN 3 MÔ HÌNH 
# ==============================

y_pred_lr = pipeline_lr.predict(X_test)

y_pred_rf = pipeline_rf.predict(X_test)

y_pred_xgb = pipeline_xgb.predict(X_test)

print("Đã dự đoán bằng 3 mô hình.")

Đã dự đoán bằng 3 mô hình.


In [16]:
# ==============================
# ĐÁNH GIÁ 3 MÔ HÌNH
# ==============================

results = []

# Linear Regression
mae_lr = mean_absolute_error(
    y_test,
    y_pred_lr
)

rmse_lr = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_lr
    )
)

r2_lr = r2_score(
    y_test,
    y_pred_lr
)

results.append([
    "Linear Regression",
    mae_lr / 1e9,
    rmse_lr / 1e9,
    r2_lr
])


# Random Forest
mae_rf = mean_absolute_error(
    y_test,
    y_pred_rf
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf
    )
)

r2_rf = r2_score(
    y_test,
    y_pred_rf
)

results.append([
    "Random Forest",
    mae_rf / 1e9,
    rmse_rf / 1e9,
    r2_rf
])


# XGBoost
mae_xgb = mean_absolute_error(
    y_test,
    y_pred_xgb
)

rmse_xgb = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb
    )
)

r2_xgb = r2_score(
    y_test,
    y_pred_xgb
)

results.append([
    "XGBoost",
    mae_xgb / 1e9,
    rmse_xgb / 1e9,
    r2_xgb
])


print("Đã đánh giá 3 mô hình.")

Đã đánh giá 3 mô hình.


In [17]:
# ==============================
# BẢNG SO SÁNH
# ==============================

results_df = pd.DataFrame(
    results,
    columns=[
        "Mô hình",
        "MAE (tỷ đồng)",
        "RMSE (tỷ đồng)",
        "R²"
    ]
)

display(
    results_df.style.format({
        "MAE (tỷ đồng)": "{:.2f}",
        "RMSE (tỷ đồng)": "{:.2f}",
        "R²": "{:.4f}"
    })
)

,Mô hình,MAE (tỷ đồng),RMSE (tỷ đồng),R²
0,Linear Regression,14.19,31.18,0.4386
1,Random Forest,7.31,24.36,0.6574
2,XGBoost,6.85,23.11,0.6915


In [18]:
# ==============================
# KẾT QUẢ
# ==============================

print("========================================")
print("       KẾT QUẢ SO SÁNH MÔ HÌNH")
print("========================================")

for result in results:
    print(
        f"{result[0]:20s} | "
        f"MAE: {result[1]:.2f} tỷ | "
        f"RMSE: {result[2]:.2f} tỷ | "
        f"R²: {result[3]:.4f}"
    )

       KẾT QUẢ SO SÁNH MÔ HÌNH
Linear Regression    | MAE: 14.19 tỷ | RMSE: 31.18 tỷ | R²: 0.4386
Random Forest        | MAE: 7.31 tỷ | RMSE: 24.36 tỷ | R²: 0.6574
XGBoost              | MAE: 6.85 tỷ | RMSE: 23.11 tỷ | R²: 0.6915


In [19]:
# ==============================
# CHỌN MÔ HÌNH
# ==============================

model_pipelines = {
    "Linear Regression": pipeline_lr,
    "Random Forest": pipeline_rf,
    "XGBoost": pipeline_xgb
}

r2_scores = {
    "Linear Regression": r2_lr,
    "Random Forest": r2_rf,
    "XGBoost": r2_xgb
}

best_model_name = max(
    r2_scores,
    key=r2_scores.get
)

best_pipeline = model_pipelines[
    best_model_name
]

print("Mô hình được chọn:")
print(best_model_name)

print(
    f"R²: {r2_scores[best_model_name]:.4f}"
)

Mô hình được chọn:
XGBoost
R²: 0.6915


In [20]:
# ==============================
# LƯU MÔ HÌNH
# ==============================

joblib.dump(
    best_pipeline,
    "../models/house_price_model.pkl"
)

print("Đã lưu model.")
print(
    f"Mô hình: {best_model_name}"
)
print(
    "File: models/house_price_model.pkl"
)

Đã lưu model.
Mô hình: XGBoost
File: models/house_price_model.pkl


In [21]:
import os

model_path = "../models/house_price_model.pkl"

if os.path.exists(model_path):
    print("Model tồn tại.")
    print(
        "Dung lượng:",
        os.path.getsize(model_path),
        "bytes"
    )
else:
    print("Không tìm thấy model.")

Model tồn tại.
Dung lượng: 913161 bytes
